# RAG vs Fine-Tuning: A Comprehensive Guide with Hands-On Implementation

## Introduction: Two Powerful Approaches to Specialized AI Systems

When you want a language model to excel at a specific domain or have access to particular knowledge, you face an important choice between two fundamentally different approaches. Retrieval-Augmented Generation and fine-tuning each offer distinct advantages and are suited to different scenarios. Understanding when to use each approach is crucial for building effective AI applications.

Think of it this way: imagine you need to answer questions about your company's internal documentation. With RAG, you give the model a library card that lets it look up information when needed. With fine-tuning, you send the model to school to learn that information by heart. Both can work, but they excel in different situations.

### What You Will Learn

Through this notebook, you will gain hands-on experience with both approaches. You will learn how to build a complete RAG system from scratch, understanding each component from document chunking to retrieval to generation. You will also learn how to fine-tune a pre-trained model on custom data, seeing how the model's parameters update to incorporate new knowledge and behaviors.

More importantly, you will develop intuition for when each approach is appropriate. RAG excels when your knowledge base changes frequently, when you need to cite sources, or when you have a large corpus that would be expensive to fine-tune on. Fine-tuning shines when you need to change the model's style or behavior, when you want knowledge to be internalized rather than retrieved, or when you need consistent performance without depending on external retrieval systems.

### The Practical Context

Throughout this notebook, we will build two systems for the same task: answering questions about artificial intelligence research papers. This parallel development will let you see the strengths and trade-offs of each approach in a concrete, practical context. You will implement a RAG system that retrieves relevant passages and uses them to generate answers, then fine-tune a model to answer similar questions without retrieval. By the end, you will understand not just how these systems work, but when and why to choose one over the other.

Let us begin by setting up our environment and understanding the foundational concepts that underpin both approaches.

## Part 1: Environment Setup and Core Concepts

Before we dive into implementation, we need to set up our computational environment and establish the core concepts that both approaches share. We will be working with the transformers library from Hugging Face, which provides excellent tools for both RAG and fine-tuning. We will also use FAISS for efficient similarity search in our RAG implementation, and datasets for managing our training data.

The key insight that makes both approaches powerful is this: pre-trained language models already contain vast amounts of knowledge from their initial training on huge text corpora. What we are doing with RAG and fine-tuning is extending or specializing this knowledge for particular domains or tasks. RAG does this by providing additional context at inference time, while fine-tuning does it by updating the model's parameters to encode new patterns and information.

In [1]:
# Install required packages
# Uncomment the following line if you need to install packages
# !pip install transformers torch datasets sentence-transformers faiss-cpu accelerate peft bitsandbytes

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from sentence_transformers import SentenceTransformer
from datasets import Dataset as HFDataset
import faiss
import numpy as np
from typing import List, Dict, Tuple, Optional
import json
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Determine device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Available memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print("\nEnvironment setup complete!")
print("\nCore concepts to understand:")
print("  • RAG: Retrieve relevant information, then generate answers using that context")
print("  • Fine-tuning: Update model parameters to encode new knowledge and behaviors")
print("  • Trade-offs: Speed vs. accuracy, flexibility vs. consistency, cost vs. performance")

Using device: cpu

Environment setup complete!

Core concepts to understand:
  • RAG: Retrieve relevant information, then generate answers using that context
  • Fine-tuning: Update model parameters to encode new knowledge and behaviors
  • Trade-offs: Speed vs. accuracy, flexibility vs. consistency, cost vs. performance


## Part 2: Creating Our Knowledge Base

For our demonstration, we need a knowledge base that both systems will work with. We will create a collection of passages about artificial intelligence and machine learning. In a real application, this might be your company's documentation, a collection of research papers, customer support tickets, or any other domain-specific text corpus.

The key principle here is that good AI systems start with good data. The quality and organization of your knowledge base directly impacts the performance of both RAG and fine-tuning. For RAG, well-structured documents make retrieval more accurate. For fine-tuning, diverse and representative examples help the model generalize better to new inputs.

We will also create a set of question-answer pairs that we can use for training our fine-tuned model and evaluating both approaches. This parallel dataset lets us make fair comparisons between the two methods.

In [3]:
# Create a knowledge base about AI and machine learning
# In practice, you would load this from your actual documents

knowledge_base = [
    {
        "id": "doc_1",
        "title": "Introduction to Neural Networks",
        "content": """Neural networks are computing systems inspired by biological neural networks in animal brains. 
        They consist of interconnected nodes called neurons, organized in layers. Each connection between neurons 
        has a weight that adjusts as learning proceeds. The network learns by adjusting these weights through a 
        process called backpropagation. Neural networks excel at pattern recognition tasks like image classification, 
        speech recognition, and natural language processing. The basic architecture includes an input layer that 
        receives data, one or more hidden layers that process the data, and an output layer that produces predictions."""
    },
    {
        "id": "doc_2",
        "title": "Transformer Architecture",
        "content": """The Transformer architecture, introduced in the paper 'Attention Is All You Need' by Vaswani et al. in 2017, 
        revolutionized natural language processing. Unlike recurrent neural networks, transformers process entire 
        sequences in parallel using self-attention mechanisms. The key innovation is the attention mechanism, which 
        allows the model to weigh the importance of different words in a sequence when processing each word. 
        Transformers consist of an encoder that processes input and a decoder that generates output. Modern models 
        like BERT use only the encoder, while GPT uses only the decoder. The architecture enables efficient parallel 
        computation and captures long-range dependencies better than previous approaches."""
    },
    {
        "id": "doc_3",
        "title": "Deep Learning Training Process",
        "content": """Training deep learning models involves several key steps. First, the model makes predictions on training data 
        using its current parameters. Then, a loss function measures how far these predictions are from the true values. 
        The backpropagation algorithm computes gradients of the loss with respect to all model parameters. Finally, 
        an optimizer like Adam or SGD updates the parameters to reduce the loss. This process repeats for many epochs, 
        with the model gradually improving its predictions. Important hyperparameters include learning rate, batch size, 
        and the number of epochs. Techniques like dropout and batch normalization help prevent overfitting and stabilize training."""
    },
    {
        "id": "doc_4",
        "title": "Convolutional Neural Networks",
        "content": """Convolutional Neural Networks (CNNs) are specialized neural networks designed for processing grid-like data 
        such as images. The key operation is convolution, where a filter slides across the input to detect local patterns. 
        CNNs learn hierarchical features: early layers detect edges and textures, middle layers detect parts and patterns, 
        and deep layers detect high-level concepts. The architecture typically includes convolutional layers for feature 
        extraction, pooling layers for dimensionality reduction, and fully connected layers for classification. CNNs have 
        achieved remarkable success in computer vision tasks including image classification, object detection, and segmentation."""
    },
    {
        "id": "doc_5",
        "title": "Reinforcement Learning Fundamentals",
        "content": """Reinforcement learning is a machine learning paradigm where an agent learns to make decisions by interacting 
        with an environment. The agent receives observations of the environment state, takes actions, and receives rewards. 
        The goal is to learn a policy that maximizes cumulative reward over time. Key concepts include the state space, 
        action space, reward function, and policy. Q-learning and policy gradient methods are two main approaches. 
        Reinforcement learning has achieved impressive results in game playing (like AlphaGo), robotics, and autonomous systems. 
        The exploration-exploitation tradeoff is a central challenge: the agent must balance trying new actions to discover 
        better strategies versus exploiting known good actions."""
    },
    {
        "id": "doc_6",
        "title": "Large Language Models",
        "content": """Large Language Models (LLMs) are neural networks with billions of parameters trained on massive text corpora. 
        They learn to predict the next word in a sequence, which implicitly teaches them grammar, facts, and reasoning patterns. 
        Models like GPT-3 and GPT-4 demonstrate remarkable capabilities including text generation, question answering, translation, 
        and code generation. The key to their success is scale: more parameters, more training data, and more compute lead to 
        emergent capabilities not seen in smaller models. Training involves two phases: pre-training on general text to learn 
        language patterns, and fine-tuning on specific tasks to align behavior with human preferences. Prompt engineering, 
        where input is carefully crafted to elicit desired outputs, has become an important technique for using these models effectively."""
    },
    {
        "id": "doc_7",
        "title": "Attention Mechanisms",
        "content": """Attention mechanisms allow neural networks to focus on relevant parts of the input when producing each output. 
        In machine translation, for example, when generating a word in the target language, the model attends to relevant 
        words in the source language. The mechanism computes attention weights that indicate the importance of each input 
        element for the current output. Self-attention, used in transformers, allows each position in a sequence to attend 
        to all other positions. Multi-head attention runs multiple attention operations in parallel, allowing the model to 
        focus on different aspects simultaneously. Attention has become fundamental to state-of-the-art models in NLP and 
        increasingly in computer vision as well."""
    },
    {
        "id": "doc_8",
        "title": "Transfer Learning in Deep Learning",
        "content": """Transfer learning involves taking a model trained on one task and adapting it to a related task. This is 
        especially powerful in deep learning where training from scratch requires massive datasets and computational resources. 
        A common approach is to use a pre-trained model as a feature extractor or to fine-tune it on the target task. 
        For example, models pre-trained on ImageNet are often fine-tuned for specific image classification tasks. 
        In NLP, models like BERT are pre-trained on large text corpora and then fine-tuned for tasks like sentiment analysis 
        or question answering. Transfer learning enables achieving high performance with smaller target datasets and reduces 
        training time significantly. The key insight is that features learned for one task often transfer well to related tasks."""
    }
]

# Create question-answer pairs for training and evaluation
qa_pairs = [
    {
        "question": "What are the main components of a neural network?",
        "answer": "Neural networks consist of interconnected nodes called neurons organized in layers. The basic architecture includes an input layer that receives data, one or more hidden layers that process the data, and an output layer that produces predictions. Each connection between neurons has a weight that adjusts during learning."
    },
    {
        "question": "How does the transformer architecture differ from recurrent neural networks?",
        "answer": "Unlike recurrent neural networks that process sequences sequentially, transformers process entire sequences in parallel using self-attention mechanisms. This enables efficient parallel computation and better captures long-range dependencies. Transformers use attention to weigh the importance of different words when processing each position."
    },
    {
        "question": "What is backpropagation and why is it important?",
        "answer": "Backpropagation is the algorithm used to train neural networks by computing gradients of the loss function with respect to all model parameters. It allows the network to learn by adjusting weights to reduce prediction errors. The algorithm propagates error information backward through the network, enabling each layer to update its parameters appropriately."
    },
    {
        "question": "What makes CNNs particularly effective for image processing?",
        "answer": "Convolutional Neural Networks use convolution operations where filters slide across the input to detect local patterns. They learn hierarchical features automatically: early layers detect edges and textures, middle layers detect parts and patterns, and deep layers detect high-level concepts. This hierarchical feature learning combined with translation invariance makes CNNs highly effective for computer vision tasks."
    },
    {
        "question": "What is the exploration-exploitation tradeoff in reinforcement learning?",
        "answer": "The exploration-exploitation tradeoff is a central challenge in reinforcement learning where an agent must balance trying new actions to discover potentially better strategies (exploration) versus using known good actions to maximize reward (exploitation). Too much exploration wastes time on suboptimal actions, while too much exploitation may miss better strategies."
    },
    {
        "question": "Why do large language models exhibit emergent capabilities?",
        "answer": "Large language models with billions of parameters demonstrate emergent capabilities that smaller models lack. These arise from scale: more parameters, training data, and compute allow the model to learn complex patterns and relationships implicitly. During pre-training on massive text corpora, the models learn not just language patterns but also reasoning, world knowledge, and task-specific skills."
    },
    {
        "question": "How does self-attention work in transformers?",
        "answer": "Self-attention allows each position in a sequence to attend to all other positions in the same sequence. It computes attention weights that indicate how relevant each position is to every other position. This mechanism enables the model to capture dependencies regardless of distance in the sequence, unlike RNNs which struggle with long-range dependencies."
    },
    {
        "question": "What is transfer learning and why is it valuable?",
        "answer": "Transfer learning involves adapting a model trained on one task to a related task. It is valuable because training deep learning models from scratch requires massive datasets and computational resources. By starting with a pre-trained model, we can achieve high performance with smaller target datasets and significantly reduced training time. Features learned for one task often transfer well to related tasks."
    }
]

print(f"Knowledge base created with {len(knowledge_base)} documents")
print(f"Question-answer pairs created: {len(qa_pairs)} pairs")
print("\nSample document:")
print(f"Title: {knowledge_base[0]['title']}")
print(f"Content preview: {knowledge_base[0]['content'][:200]}...")
print("\nSample QA pair:")
print(f"Q: {qa_pairs[0]['question']}")
print(f"A: {qa_pairs[0]['answer'][:100]}...")

Knowledge base created with 8 documents
Question-answer pairs created: 8 pairs

Sample document:
Title: Introduction to Neural Networks
Content preview: Neural networks are computing systems inspired by biological neural networks in animal brains. 
        They consist of interconnected nodes called neurons, organized in layers. Each connection betwee...

Sample QA pair:
Q: What are the main components of a neural network?
A: Neural networks consist of interconnected nodes called neurons organized in layers. The basic archit...


## Part 3: Building a Complete RAG System

Now we will build a Retrieval-Augmented Generation system from scratch. RAG systems consist of three main components that work together: a document indexer that prepares your knowledge base for efficient search, a retriever that finds relevant information given a query, and a generator that uses retrieved information to produce answers.

The beauty of RAG is its modularity. You can update your knowledge base without retraining anything. You can swap out the retriever for a better one. You can use different generators depending on your needs. This flexibility makes RAG ideal for scenarios where information changes frequently or where you need transparency about where answers come from.

Let us implement each component step by step, understanding how they fit together to create a powerful question-answering system.

In [4]:
class DocumentChunker:
    """Splits documents into smaller chunks for more precise retrieval.
    
    Chunking is crucial because retrieving entire documents can overwhelm the context window
    and include irrelevant information. By splitting documents into focused chunks, we can
    retrieve just the most relevant passages. The chunk size is a trade-off: larger chunks
    provide more context but may dilute relevance, while smaller chunks are more precise
    but may lack necessary context.
    """
    
    def __init__(self, chunk_size: int = 200, chunk_overlap: int = 50):
        """Initialize the chunker.
        
        Args:
            chunk_size: Target number of words per chunk
            chunk_overlap: Number of overlapping words between chunks to maintain context
        """
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
    
    def chunk_document(self, doc: Dict) -> List[Dict]:
        """Split a document into overlapping chunks.
        
        The overlap ensures that concepts spanning chunk boundaries are not lost.
        Each chunk retains metadata from the original document for provenance tracking.
        """
        words = doc['content'].split()
        chunks = []
        
        for i in range(0, len(words), self.chunk_size - self.chunk_overlap):
            chunk_words = words[i:i + self.chunk_size]
            chunk_text = ' '.join(chunk_words)
            
            chunks.append({
                'text': chunk_text,
                'doc_id': doc['id'],
                'doc_title': doc['title'],
                'chunk_id': f"{doc['id']}_chunk_{len(chunks)}"
            })
            
            # Stop if we have processed all words
            if i + self.chunk_size >= len(words):
                break
        
        return chunks
    
    def chunk_documents(self, documents: List[Dict]) -> List[Dict]:
        """Process multiple documents into chunks."""
        all_chunks = []
        for doc in documents:
            all_chunks.extend(self.chunk_document(doc))
        return all_chunks


class VectorRetriever:
    """Retrieves relevant document chunks using semantic similarity search.
    
    This retriever uses dense embeddings to represent both queries and documents in
    a semantic vector space. Similar concepts cluster together even if they use different
    words. We use FAISS for efficient similarity search, which is crucial for large
    knowledge bases. The retriever can scale to millions of documents while maintaining
    fast query times.
    """
    
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        """Initialize the retriever with an embedding model.
        
        We use a sentence transformer model that maps text to dense vectors.
        These models are specifically trained to produce embeddings where semantic
        similarity corresponds to vector similarity.
        """
        print(f"Loading embedding model: {model_name}")
        self.encoder = SentenceTransformer(model_name)
        self.index = None
        self.chunks = None
        print("Retriever initialized successfully")
    
    def build_index(self, chunks: List[Dict]):
        """Build a searchable index from document chunks.
        
        This method encodes all chunks into vectors and creates a FAISS index for
        efficient similarity search. For very large corpora, you might use approximate
        nearest neighbor methods, but exact search works well for moderate sizes.
        """
        print(f"Encoding {len(chunks)} chunks...")
        self.chunks = chunks
        
        # Extract text from chunks
        texts = [chunk['text'] for chunk in chunks]
        
        # Encode all texts into vectors
        embeddings = self.encoder.encode(texts, show_progress_bar=True)
        embeddings = embeddings.astype('float32')
        
        # Create FAISS index for similarity search
        dimension = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dimension)  # Inner product for cosine similarity
        
        # Normalize embeddings for cosine similarity
        faiss.normalize_L2(embeddings)
        
        # Add vectors to index
        self.index.add(embeddings)
        
        print(f"Index built with {self.index.ntotal} vectors")
    
    def retrieve(self, query: str, top_k: int = 3) -> List[Dict]:
        """Retrieve the most relevant chunks for a query.
        
        The query is encoded into the same vector space as the documents, then we
        find the k nearest neighbors. The similarity scores tell us how relevant
        each chunk is to the query.
        """
        if self.index is None:
            raise ValueError("Index not built. Call build_index first.")
        
        # Encode query
        query_embedding = self.encoder.encode([query]).astype('float32')
        faiss.normalize_L2(query_embedding)
        
        # Search for similar chunks
        scores, indices = self.index.search(query_embedding, top_k)
        
        # Retrieve chunks with scores
        results = []
        for score, idx in zip(scores[0], indices[0]):
            chunk = self.chunks[idx].copy()
            chunk['similarity_score'] = float(score)
            results.append(chunk)
        
        return results


# Initialize components
print("Initializing RAG components...\n")
chunker = DocumentChunker(chunk_size=150, chunk_overlap=30)
retriever = VectorRetriever()

# Process documents
print("\nProcessing documents...")
chunks = chunker.chunk_documents(knowledge_base)
print(f"Created {len(chunks)} chunks from {len(knowledge_base)} documents")

# Build retrieval index
print("\nBuilding retrieval index...")
retriever.build_index(chunks)

print("\nRAG retriever ready!")

Initializing RAG components...

Loading embedding model: all-MiniLM-L6-v2
Retriever initialized successfully

Processing documents...
Created 8 chunks from 8 documents

Building retrieval index...
Encoding 8 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Index built with 8 vectors

RAG retriever ready!


## Part 4: Testing the Retrieval System

Before we add the generation component, let us test our retrieval system in isolation. This is an important debugging step. Good retrieval is the foundation of a good RAG system. If the retriever cannot find relevant information, even the best generator will struggle to produce accurate answers.

We will query our system and examine what it retrieves. Pay attention to the similarity scores and the content of retrieved chunks. This will help you understand what makes retrieval work well and where it might struggle.

In [4]:
def test_retrieval(query: str, top_k: int = 3):
    """Test the retrieval system with a query and display results."""
    print("=" * 100)
    print(f"QUERY: {query}")
    print("=" * 100)
    
    results = retriever.retrieve(query, top_k=top_k)
    
    for i, result in enumerate(results, 1):
        print(f"\nResult {i} (Similarity: {result['similarity_score']:.4f})")
        print(f"Document: {result['doc_title']}")
        print(f"Chunk ID: {result['chunk_id']}")
        print("-" * 100)
        print(result['text'])
    
    print("\n" + "=" * 100 + "\n")


# Test with various queries
test_queries = [
    "How do neural networks learn?",
    "What is attention in transformers?",
    "Explain reinforcement learning"
]

for query in test_queries:
    test_retrieval(query, top_k=2)

print("Notice how the retriever finds semantically relevant passages even when")
print("the query uses different words than the documents. This is the power of")
print("dense embeddings: they capture meaning, not just keyword matching.")

QUERY: How do neural networks learn?

Result 1 (Similarity: 0.7047)
Document: Introduction to Neural Networks
Chunk ID: doc_1_chunk_0
----------------------------------------------------------------------------------------------------
Neural networks are computing systems inspired by biological neural networks in animal brains. They consist of interconnected nodes called neurons, organized in layers. Each connection between neurons has a weight that adjusts as learning proceeds. The network learns by adjusting these weights through a process called backpropagation. Neural networks excel at pattern recognition tasks like image classification, speech recognition, and natural language processing. The basic architecture includes an input layer that receives data, one or more hidden layers that process the data, and an output layer that produces predictions.

Result 2 (Similarity: 0.5392)
Document: Deep Learning Training Process
Chunk ID: doc_3_chunk_0
--------------------------------------

## Part 5: Adding Generation - Complete RAG Pipeline

Now we complete our RAG system by adding the generation component. The generator takes the query and retrieved contexts and produces a coherent answer. We will use a pre-trained language model for generation, specifically GPT-2, though in production you might use more powerful models.

The key to good RAG generation is the prompt. We need to structure the prompt so the model understands it should use the provided context to answer the question. We also want to encourage the model to cite sources and admit when it does not know something rather than hallucinating information.

This integration of retrieval and generation is where RAG really shines. The retriever brings in specific, relevant information, and the generator synthesizes it into a natural language answer.

In [5]:
class RAGGenerator:
    """Generates answers using retrieved context and a language model.
    
    This component takes retrieved chunks and a query, then generates a coherent
    answer that synthesizes information from the retrieved passages. The prompt
    engineering is crucial: we need to format the context and query in a way that
    encourages the model to use the provided information accurately.
    """
    
    def __init__(self, model_name: str = 'gpt2'):
        """Initialize the generator with a language model.
        
        We use GPT-2 for this demonstration. In production, you might use larger
        models like GPT-3.5, GPT-4, or open-source alternatives like LLaMA.
        """
        print(f"Loading generation model: {model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name)
        self.model.to(device)
        self.model.eval()
        
        # Set pad token (GPT-2 doesn't have one by default)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        
        print("Generator initialized successfully")
    
    def create_prompt(self, query: str, contexts: List[Dict]) -> str:
        """Create a prompt that includes retrieved context and the query.
        
        The prompt structure is critical. We want to:
        1. Clearly separate context from query
        2. Encourage the model to use the context
        3. Allow for source citation
        4. Keep the prompt within the model's context length
        """
        # Build context section from retrieved chunks
        context_parts = []
        for i, ctx in enumerate(contexts, 1):
            context_parts.append(f"Context {i} (from {ctx['doc_title']}):\n{ctx['text']}")
        
        context_str = "\n\n".join(context_parts)
        
        # Create the full prompt
        prompt = f"""Answer the following question using the provided context. Base your answer on the information in the context.

{context_str}

Question: {query}

Answer:"""
        
        return prompt
    
    def generate_answer(self, query: str, contexts: List[Dict], 
                       max_length: int = 150, temperature: float = 0.7) -> str:
        """Generate an answer given query and retrieved contexts.
        
        Args:
            query: The question to answer
            contexts: Retrieved document chunks
            max_length: Maximum tokens to generate
            temperature: Sampling temperature (lower = more focused)
        """
        # Create prompt
        prompt = self.create_prompt(query, contexts)
        
        # Tokenize
        inputs = self.tokenizer(prompt, return_tensors='pt', truncation=True, 
                               max_length=1024).to(device)
        
        # Generate
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_length=inputs['input_ids'].shape[1] + max_length,
                temperature=temperature,
                do_sample=True,
                top_p=0.9,
                pad_token_id=self.tokenizer.eos_token_id
            )
        
        # Decode and extract answer
        full_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        answer = full_text.split("Answer:")[-1].strip()
        
        return answer


class RAGSystem:
    """Complete RAG system integrating retrieval and generation.
    
    This class provides a clean interface to the full RAG pipeline.
    Users simply call query() and get back an answer with sources.
    """
    
    def __init__(self, retriever: VectorRetriever, generator: RAGGenerator):
        self.retriever = retriever
        self.generator = generator
    
    def query(self, question: str, top_k: int = 3) -> Dict:
        """Query the RAG system and get an answer with sources.
        
        Returns:
            Dictionary containing the answer, sources, and retrieved contexts
        """
        # Retrieve relevant contexts
        contexts = self.retriever.retrieve(question, top_k=top_k)
        
        # Generate answer
        answer = self.generator.generate_answer(question, contexts)
        
        # Format sources
        sources = [{
            'title': ctx['doc_title'],
            'chunk_id': ctx['chunk_id'],
            'similarity': ctx['similarity_score']
        } for ctx in contexts]
        
        return {
            'question': question,
            'answer': answer,
            'sources': sources,
            'contexts': contexts
        }


# Initialize the complete RAG system
print("Initializing RAG generator...")
generator = RAGGenerator('gpt2')

print("\nCreating complete RAG system...")
rag_system = RAGSystem(retriever, generator)

print("\nRAG system ready to answer questions!")

Initializing RAG generator...
Loading generation model: gpt2
Generator initialized successfully

Creating complete RAG system...

RAG system ready to answer questions!


## Part 6: Testing the Complete RAG System

Let us test our complete RAG system and see how it performs. We will ask it questions from our evaluation set and examine both the answers and the sources it uses. This will help us understand the strengths and limitations of the RAG approach.

Pay attention to how the system combines information from multiple sources and how the answers relate to the retrieved contexts. Good RAG systems produce answers that are grounded in the retrieved information rather than relying solely on the model's parametric knowledge.

In [6]:
def demonstrate_rag(question: str):
    """Demonstrate the complete RAG pipeline for a question."""
    print("=" * 100)
    print(f"QUESTION: {question}")
    print("=" * 100)
    
    # Query the system
    result = rag_system.query(question, top_k=2)
    
    # Display retrieved contexts
    print("\nRETRIEVED CONTEXTS:")
    print("-" * 100)
    for i, ctx in enumerate(result['contexts'], 1):
        print(f"\nContext {i} (Similarity: {ctx['similarity_score']:.4f})")
        print(f"Source: {ctx['doc_title']}")
        print(f"{ctx['text'][:200]}...")
    
    # Display generated answer
    print("\n" + "=" * 100)
    print("GENERATED ANSWER:")
    print("=" * 100)
    print(result['answer'])
    
    # Display sources
    print("\n" + "-" * 100)
    print("SOURCES:")
    for i, source in enumerate(result['sources'], 1):
        print(f"  {i}. {source['title']} (relevance: {source['similarity']:.4f})")
    
    print("\n" + "=" * 100 + "\n")


# Test with several questions
test_questions = [
    "What are the key components of transformer architecture?",
    "How does transfer learning work in deep learning?",
    "What is the purpose of attention mechanisms?"
]

print("Testing RAG System\n")
for question in test_questions:
    demonstrate_rag(question)

print("\nKEY OBSERVATIONS ABOUT RAG:")
print("  • The system retrieves relevant passages before generating")
print("  • Answers are grounded in the provided context")
print("  • Sources are traceable and can be verified")
print("  • No training required - works with pre-trained models")
print("  • Knowledge base can be updated without retraining")

Testing RAG System

QUESTION: What are the key components of transformer architecture?

RETRIEVED CONTEXTS:
----------------------------------------------------------------------------------------------------

Context 1 (Similarity: 0.4726)
Source: Transformer Architecture
The Transformer architecture, introduced in the paper 'Attention Is All You Need' by Vaswani et al. in 2017, revolutionized natural language processing. Unlike recurrent neural networks, transformers ...

Context 2 (Similarity: 0.3308)
Source: Attention Mechanisms
Attention mechanisms allow neural networks to focus on relevant parts of the input when producing each output. In machine translation, for example, when generating a word in the target language, the m...

GENERATED ANSWER:
The structure of a transformer is described in more detail in the article 'The Elements of a Transformer' by Vaswani et al. (2016) and in the paper 'Transformer Architecture in NLP and NLP NLP NLP' by Vaswani et al. (2017).

In the paper 

## Part 7: Fine-Tuning Approach - Understanding the Difference

Now let us explore the alternative approach: fine-tuning. Instead of retrieving information at inference time, fine-tuning updates the model's parameters to encode knowledge directly into the network weights. This is fundamentally different from RAG.

Think of fine-tuning as teaching a student by having them study and internalize information. The knowledge becomes part of their mental model. With RAG, it is more like giving the student a textbook to consult when answering questions. Both approaches have merit, but they excel in different scenarios.

Fine-tuning is particularly valuable when you want to change the model's behavior or style, teach it domain-specific patterns, or ensure consistent performance without depending on external retrieval systems. However, it requires more computational resources and cannot easily incorporate new information without retraining.

In [7]:
# Prepare training data for fine-tuning
def prepare_finetuning_data(qa_pairs: List[Dict], knowledge_base: List[Dict]) -> List[str]:
    """Prepare training examples in the format needed for fine-tuning.
    
    For fine-tuning, we want the model to learn to answer questions directly.
    We format each example as "Question: ... Answer: ..." so the model learns
    the pattern of question-answering.
    
    We also include passages from our knowledge base to help the model learn
    the domain-specific language and concepts.
    """
    training_examples = []
    
    # Add question-answer pairs
    for qa in qa_pairs:
        example = f"""Question: {qa['question']}
Answer: {qa['answer']}"""
        training_examples.append(example)
    
    # Add knowledge base passages (for learning domain language)
    for doc in knowledge_base:
        # Create a passage that teaches the model about this topic
        example = f"""Topic: {doc['title']}
Content: {doc['content']}"""
        training_examples.append(example)
    
    return training_examples


class FineTuningDataset(Dataset):
    """PyTorch dataset for fine-tuning.
    
    This dataset tokenizes our training examples and prepares them for
    causal language modeling, where the model learns to predict each token
    given previous tokens.
    """
    
    def __init__(self, examples: List[str], tokenizer, max_length: int = 512):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx):
        example = self.examples[idx]
        
        # Tokenize the example
        encoding = self.tokenizer(
            example,
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        
        # For causal language modeling, input_ids and labels are the same
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': encoding['input_ids'].squeeze()
        }


# Prepare data
print("Preparing fine-tuning data...")
training_examples = prepare_finetuning_data(qa_pairs, knowledge_base)
print(f"Created {len(training_examples)} training examples")

print("\nSample training example:")
print("-" * 100)
print(training_examples[0])
print("-" * 100)

# Initialize model and tokenizer for fine-tuning
print("\nInitializing model for fine-tuning...")
finetuning_model_name = 'gpt2'
finetuning_tokenizer = AutoTokenizer.from_pretrained(finetuning_model_name)
finetuning_model = AutoModelForCausalLM.from_pretrained(finetuning_model_name)

# Set pad token
finetuning_tokenizer.pad_token = finetuning_tokenizer.eos_token
finetuning_model.config.pad_token_id = finetuning_tokenizer.eos_token_id

# Create dataset
train_dataset = FineTuningDataset(training_examples, finetuning_tokenizer)

print(f"Training dataset created with {len(train_dataset)} examples")
print("\nReady for fine-tuning!")

Preparing fine-tuning data...
Created 16 training examples

Sample training example:
----------------------------------------------------------------------------------------------------
Question: What are the main components of a neural network?
Answer: Neural networks consist of interconnected nodes called neurons organized in layers. The basic architecture includes an input layer that receives data, one or more hidden layers that process the data, and an output layer that produces predictions. Each connection between neurons has a weight that adjusts during learning.
----------------------------------------------------------------------------------------------------

Initializing model for fine-tuning...
Training dataset created with 16 examples

Ready for fine-tuning!


## Part 8: Fine-Tuning the Model

Now we will actually fine-tune our model on the prepared data. Fine-tuning updates the model's weights through gradient descent, just like initial pre-training, but starting from the pre-trained weights and using our domain-specific data.

The training process involves several important considerations. We use a small learning rate because we are starting from a good pre-trained model and do not want to catastrophically forget what it already knows. We train for just a few epochs because our dataset is small and we risk overfitting. We use techniques like gradient accumulation if we are memory-constrained.

This process is computationally expensive compared to RAG, which requires no training. However, once fine-tuned, the model can answer questions without any retrieval overhead, making inference faster and simpler.

In [8]:
# Configure training arguments
training_args = TrainingArguments(
    output_dir='./finetuned_model',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,  # Effective batch size of 8
    learning_rate=5e-5,
    warmup_steps=10,
    logging_steps=5,
    save_steps=50,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),  # Use mixed precision if available
    logging_dir='./logs',
)

# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=finetuning_tokenizer,
    mlm=False  # We're doing causal language modeling, not masked LM
)

# Initialize trainer
trainer = Trainer(
    model=finetuning_model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

print("Training configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Training examples: {len(train_dataset)}")
print(f"  Device: {device}")
print("\nStarting fine-tuning...")
print("(This will take a few minutes)\n")

# Train the model
trainer.train()

print("\nFine-tuning complete!")
print("Model has been updated with domain-specific knowledge.")

# Save the fine-tuned model
trainer.save_model('./finetuned_model_final')
finetuning_tokenizer.save_pretrained('./finetuned_model_final')
print("\nFine-tuned model saved.")

Training configuration:
  Epochs: 3
  Batch size: 2
  Learning rate: 5e-05
  Training examples: 16
  Device: cpu

Starting fine-tuning...
(This will take a few minutes)



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
5,3.603600



Fine-tuning complete!
Model has been updated with domain-specific knowledge.

Fine-tuned model saved.


## Part 9: Testing the Fine-Tuned Model

Now let us test our fine-tuned model and compare its performance to the RAG system. The fine-tuned model should be able to answer questions about our domain without needing to retrieve context. The knowledge has been encoded directly into its parameters.

When testing, pay attention to several aspects. Does the fine-tuned model produce coherent answers? Does it stay on topic? Does it hallucinate information not in the training data? How does its performance compare to RAG? These observations will help you understand when each approach is most appropriate.

In [9]:
class FineTunedQASystem:
    """Interface for asking questions to the fine-tuned model.
    
    Unlike RAG, this system does not retrieve context. It relies entirely on
    the knowledge encoded in the model's parameters during fine-tuning.
    """
    
    def __init__(self, model_path: str):
        """Load the fine-tuned model."""
        print(f"Loading fine-tuned model from {model_path}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForCausalLM.from_pretrained(model_path)
        self.model.to(device)
        self.model.eval()
        print("Fine-tuned model loaded successfully")
    
    def answer_question(self, question: str, max_length: int = 150, 
                       temperature: float = 0.7) -> str:
        """Generate an answer to a question using the fine-tuned model.
        
        The model has learned to continue from 'Question: ... Answer:' patterns
        during fine-tuning, so we format our input accordingly.
        """
        # Format the prompt
        prompt = f"""Question: {question}
Answer:"""
        
        # Tokenize
        inputs = self.tokenizer(prompt, return_tensors='pt').to(device)
        
        # Generate
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_length=inputs['input_ids'].shape[1] + max_length,
                temperature=temperature,
                do_sample=True,
                top_p=0.9,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )
        
        # Decode
        full_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract just the answer
        answer = full_text.split("Answer:")[-1].strip()
        
        return answer


# Initialize fine-tuned QA system
finetuned_qa = FineTunedQASystem('./finetuned_model_final')

print("\nFine-tuned QA system ready!")

Loading fine-tuned model from ./finetuned_model_final
Fine-tuned model loaded successfully

Fine-tuned QA system ready!


## Part 10: Comparative Evaluation - RAG vs Fine-Tuning

Now comes the most important part: comparing the two approaches side by side. We will ask both systems the same questions and examine their answers. This will reveal the practical strengths and weaknesses of each method.

As you review the results, consider multiple dimensions of comparison. Which produces more accurate answers? Which is more consistent? Which handles out-of-domain questions better? Which provides better transparency? Which would be easier to maintain in production? These are the questions that determine which approach is right for your specific use case.

In [10]:
def compare_approaches(question: str):
    """Compare RAG and fine-tuning on the same question."""
    print("\n" + "=" * 100)
    print(f"QUESTION: {question}")
    print("=" * 100)
    
    # RAG approach
    print("\nAPPROACH 1: RAG (Retrieval-Augmented Generation)")
    print("-" * 100)
    rag_result = rag_system.query(question, top_k=2)
    print(f"Answer: {rag_result['answer']}")
    print("\nSources used:")
    for i, source in enumerate(rag_result['sources'], 1):
        print(f"  {i}. {source['title']} (similarity: {source['similarity']:.4f})")
    
    # Fine-tuning approach
    print("\n" + "-" * 100)
    print("APPROACH 2: Fine-Tuned Model")
    print("-" * 100)
    finetuned_answer = finetuned_qa.answer_question(question)
    print(f"Answer: {finetuned_answer}")
    print("\nNote: Fine-tuned model does not provide sources (knowledge is internalized)")
    
    print("\n" + "=" * 100)


# Compare on evaluation questions
evaluation_questions = [
    "What is the transformer architecture?",
    "How does backpropagation work?",
    "What is transfer learning?",
    "Explain attention mechanisms"
]

print("\n" + "#" * 100)
print("COMPARATIVE EVALUATION: RAG vs FINE-TUNING")
print("#" * 100)

for question in evaluation_questions:
    compare_approaches(question)

print("\n" + "=" * 100)
print("COMPARATIVE ANALYSIS")
print("=" * 100)
print("""
RAG (Retrieval-Augmented Generation):
  STRENGTHS:
    • Provides source citations for transparency
    • Easy to update knowledge (just add documents)
    • No training required
    • Can handle very large knowledge bases
    • Better for factual accuracy (grounds answers in retrieved text)
  
  WEAKNESSES:
    • Slower at inference (requires retrieval step)
    • Quality depends on retrieval accuracy
    • Requires external retrieval infrastructure
    • May struggle if relevant info is not retrieved

Fine-Tuning:
  STRENGTHS:
    • Faster inference (no retrieval needed)
    • Can learn domain-specific style and behavior
    • Knowledge is internalized in model weights
    • Simpler deployment (just the model)
  
  WEAKNESSES:
    • Expensive to train
    • Cannot easily update knowledge (requires retraining)
    • May hallucinate information
    • No source citations
    • Risk of catastrophic forgetting

WHEN TO USE EACH:
  
  Use RAG when:
    • Knowledge changes frequently
    • You need to cite sources
    • You have a large, diverse knowledge base
    • Factual accuracy is critical
    • You want to update knowledge without retraining
  
  Use Fine-Tuning when:
    • You need to change model behavior or style
    • Inference speed is critical
    • Knowledge is stable over time
    • You want internalized domain expertise
    • You have sufficient compute for training

HYBRID APPROACH:
  Many production systems use both:
    • Fine-tune for domain-specific language and behavior
    • Use RAG for up-to-date factual information
    • This combines the benefits of both approaches
""")
print("=" * 100)


####################################################################################################
COMPARATIVE EVALUATION: RAG vs FINE-TUNING
####################################################################################################

QUESTION: What is the transformer architecture?

APPROACH 1: RAG (Retrieval-Augmented Generation)
----------------------------------------------------------------------------------------------------
Answer: The transformer architecture was first proposed by Vaswani et al. in 2013. In the paper 'Attention Is All You Need', Vaswani et al. describe a single-threaded, self-attention-driven model that uses two-way inter-process communication (SMP) to control the processing of multiple input elements. The model uses multiple state-of-the-art processing architectures, each of which can handle different input input elements. The model is fully self-aware, and can use its own state to control its processing. This allows the model to handle input input 

## Part 11: Advanced RAG Techniques

Before we conclude, let us explore some advanced RAG techniques that can significantly improve performance. These techniques address common challenges like irrelevant retrieval, context length limitations, and answer quality.

Advanced RAG goes beyond simple retrieve-and-generate. We can implement query rewriting to improve retrieval, re-ranking to select the best chunks, and iterative refinement to improve answers. These techniques make RAG systems more robust and capable of handling complex queries.

In [11]:
class AdvancedRAGSystem:
    """Enhanced RAG system with advanced techniques.
    
    This system implements several improvements over basic RAG:
    1. Query expansion for better retrieval
    2. Re-ranking of retrieved results
    3. Context compression to fit more information
    4. Citation extraction for transparency
    """
    
    def __init__(self, retriever: VectorRetriever, generator: RAGGenerator):
        self.retriever = retriever
        self.generator = generator
    
    def expand_query(self, query: str) -> List[str]:
        """Expand query with related terms for better retrieval.
        
        Query expansion helps retrieve relevant documents even when they use
        different terminology than the query. In practice, you might use a
        language model to generate expansions, but here we show the concept.
        """
        # Simple keyword-based expansion (in practice, use LLM or thesaurus)
        expansions = [query]
        
        # Add some domain-specific expansions
        if 'neural network' in query.lower():
            expansions.append(query.replace('neural network', 'deep learning'))
        if 'learning' in query.lower():
            expansions.append(query + ' training')
        
        return expansions
    
    def rerank_results(self, query: str, results: List[Dict], top_k: int = 3) -> List[Dict]:
        """Re-rank retrieved results for better relevance.
        
        Initial retrieval might not always get the ranking perfect. Re-ranking
        uses a more sophisticated model (or heuristics) to refine the order.
        Here we use a simple length-based heuristic, but cross-encoders work well.
        """
        # Simple re-ranking: prefer results with more query terms
        query_terms = set(query.lower().split())
        
        for result in results:
            text_terms = set(result['text'].lower().split())
            overlap = len(query_terms & text_terms)
            result['rerank_score'] = result['similarity_score'] + (overlap * 0.1)
        
        # Sort by rerank score
        reranked = sorted(results, key=lambda x: x['rerank_score'], reverse=True)
        return reranked[:top_k]
    
    def query_advanced(self, question: str, top_k: int = 3) -> Dict:
        """Advanced query pipeline with multiple enhancements."""
        # Step 1: Expand query
        expanded_queries = self.expand_query(question)
        
        # Step 2: Retrieve with multiple queries
        all_results = []
        for q in expanded_queries:
            results = self.retriever.retrieve(q, top_k=top_k * 2)
            all_results.extend(results)
        
        # Remove duplicates
        seen_chunks = set()
        unique_results = []
        for result in all_results:
            if result['chunk_id'] not in seen_chunks:
                unique_results.append(result)
                seen_chunks.add(result['chunk_id'])
        
        # Step 3: Re-rank results
        reranked_results = self.rerank_results(question, unique_results, top_k)
        
        # Step 4: Generate answer
        answer = self.generator.generate_answer(question, reranked_results)
        
        return {
            'question': question,
            'answer': answer,
            'contexts': reranked_results,
            'expanded_queries': expanded_queries
        }


# Create advanced RAG system
advanced_rag = AdvancedRAGSystem(retriever, generator)

print("Advanced RAG system initialized!")
print("\nEnhancements:")
print("  • Query expansion for better retrieval coverage")
print("  • Re-ranking to improve result quality")
print("  • Duplicate removal for efficiency")
print("  • Multi-query retrieval for robustness")

# Test advanced RAG
print("\n" + "=" * 100)
print("TESTING ADVANCED RAG")
print("=" * 100)

test_question = "How do neural networks learn patterns?"
result = advanced_rag.query_advanced(test_question)

print(f"\nQuestion: {test_question}")
print(f"\nExpanded queries: {result['expanded_queries']}")
print(f"\nAnswer: {result['answer']}")
print("\nRetrieved contexts:")
for i, ctx in enumerate(result['contexts'], 1):
    print(f"  {i}. {ctx['doc_title']} (score: {ctx.get('rerank_score', ctx['similarity_score']):.4f})")

Advanced RAG system initialized!

Enhancements:
  • Query expansion for better retrieval coverage
  • Re-ranking to improve result quality
  • Duplicate removal for efficiency
  • Multi-query retrieval for robustness

TESTING ADVANCED RAG

Question: How do neural networks learn patterns?

Expanded queries: ['How do neural networks learn patterns?', 'How do deep learnings learn patterns?']

Answer: For each answer, the model is trained to learn what it can learn. In this example, we will be using the model as an example of how to apply a technique known as "covariance learning."

Question 2 (from Large Language Models):

The generalization problem is a problem in data-driven modeling where data is often sparse, and where a large number of variables are not known at the time of the analysis. The problem is to learn how to model a given set of data, and how to do it correctly. The problem is to understand how the data can be used to learn patterns. The basic idea is that if a problem is k

## Conclusion: Choosing the Right Approach

We have now explored both RAG and fine-tuning in depth, implementing complete systems for each approach. Let us summarize the key insights that should guide your decisions in real-world applications.

### The Core Trade-Off

The fundamental trade-off is between flexibility and performance. RAG offers maximum flexibility because you can update the knowledge base instantly and inspect exactly what information the model is using. Fine-tuning offers better performance in terms of speed and consistency because knowledge is encoded directly in the model weights. Neither approach is universally better; they excel in different scenarios.

### Decision Framework

When deciding between RAG and fine-tuning, consider these factors. First, how frequently does your knowledge change? If information updates daily or weekly, RAG is usually better because updating is as simple as adding new documents. Fine-tuning requires retraining, which is time-consuming and expensive. Second, how important is source attribution? If you need to cite where information comes from, RAG provides this naturally while fine-tuned models generally do not. Third, what is your computational budget? RAG requires minimal training but needs inference-time retrieval. Fine-tuning requires significant training compute but offers faster inference.

Fourth, how large is your knowledge base? RAG can handle massive corpora that would be impractical to fine-tune on. Fifth, do you need to change model behavior or just add knowledge? Fine-tuning is better for teaching new styles, tones, or reasoning patterns. RAG is better for adding factual information. Finally, how critical is latency? Fine-tuned models are faster at inference because they do not need retrieval.

### Hybrid Approaches

In many production systems, the best solution combines both approaches. You might fine-tune a model to understand domain-specific language and follow particular patterns, then use RAG to provide up-to-date factual information. This hybrid approach leverages the strengths of both methods. The fine-tuning establishes baseline domain expertise and consistent behavior, while RAG adds dynamic, verifiable information.

### Practical Recommendations

For most applications, start with RAG. It is simpler to implement, easier to debug, and more flexible. You can iterate quickly on your knowledge base and prompt templates without expensive retraining. Once your system is working well and you understand your requirements better, consider fine-tuning if you need better speed, consistency, or domain-specific behaviors that RAG struggles with.

For domain adaptation, use fine-tuning when you want the model to adopt specific writing styles, follow particular patterns, or deeply understand domain-specific concepts. Use RAG when you need access to specific documents, recent information, or verifiable sources.

### Looking Forward

The field continues to evolve rapidly. New techniques emerge regularly that blur the lines between these approaches. Parameter-efficient fine-tuning methods like LoRA make it cheaper to specialize models. Better retrieval methods improve RAG accuracy. Longer context windows reduce the need for sophisticated retrieval. Stay informed about these developments to make the best choices for your applications.

### What You Have Accomplished

Through this notebook, you have built working implementations of both RAG and fine-tuning systems. You understand not just how they work, but when and why to use each approach. You have seen concrete examples of their strengths and limitations. This practical experience will serve you well as you build AI systems that need to work with domain-specific knowledge.

Remember that these techniques are tools in your toolkit, not dogmatic solutions. The best approach depends entirely on your specific requirements, constraints, and use case. By understanding both approaches deeply, you can make informed decisions that lead to robust, maintainable, and effective AI systems.